# LoRA Rank Sweep on GPT-2 Medium / E2E

This notebook runs the paper-inspired rank ablation for our LoRA GPT-2 Medium E2E project.

Paper motivation: Section 7.2 and Appendix H.2 ask what rank `r` is needed for LoRA. For GPT-2 Medium on E2E, the paper reports that validation loss improves up to larger ranks, but test BLEU peaks around a small rank (`r=4`).

This notebook will:

- clone/update the project,
- download and preprocess E2E,
- generate one config per rank,
- train separate LoRA adapters for each rank,
- generate and evaluate each adapter,
- save a rank-sweep CSV and plot,
- back up all run folders to Drive.

Full sweep is expensive. Use `MAX_TRAIN_STEPS=2000` for a smoke/pilot sweep, and `MAX_TRAIN_STEPS=None` for full 5-epoch runs.

## 1. Check GPU

In [ ]:
!nvidia-smi

import torch
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

## 2. Clone Or Update Repo

In [ ]:
from getpass import getpass
from pathlib import Path
import os
import subprocess

REPO_OWNER = 'justinlxiang'
REPO_NAME = 'CS4782-final-project'
BRANCH = 'main'
PROJECT_DIR = Path('/content') / REPO_NAME
WORK_DIR = PROJECT_DIR / 'code'

token = getpass('GitHub token, or press Enter for public clone: ')
repo_url = f'https://github.com/{REPO_OWNER}/{REPO_NAME}.git'
if token:
    repo_url = f'https://{token}@github.com/{REPO_OWNER}/{REPO_NAME}.git'

if PROJECT_DIR.exists():
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, repo_url, str(PROJECT_DIR)], check=True)

os.chdir(WORK_DIR)
print('working directory:', Path.cwd())
!git log --oneline -3

## 3. Install Dependencies

In [ ]:
!pip install -q -r requirements.txt

## 4. Mount Drive

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')

DRIVE_SWEEP_DIR = Path('/content/drive/MyDrive/lora_rank_sweep')
DRIVE_SWEEP_DIR.mkdir(parents=True, exist_ok=True)
print('Drive sweep dir:', DRIVE_SWEEP_DIR)

## 5. Download And Preprocess E2E

In [ ]:
!mkdir -p ../data/raw/e2e
!curl -L -o ../data/raw/e2e/train.txt https://raw.githubusercontent.com/microsoft/LoRA/main/examples/NLG/data/e2e/train.txt
!curl -L -o ../data/raw/e2e/valid.txt https://raw.githubusercontent.com/microsoft/LoRA/main/examples/NLG/data/e2e/valid.txt
!curl -L -o ../data/raw/e2e/test.txt https://raw.githubusercontent.com/microsoft/LoRA/main/examples/NLG/data/e2e/test.txt
!python scripts/prepare_e2e.py --config configs/e2e_gpt2_medium_lora.yaml
!wc -l ../data/raw/e2e/*.txt ../data/processed/e2e_gpt2/*.jsonl

## 6. Rank Sweep Settings

Recommended options:

- Quick pilot: `RANKS = [1, 4, 16]`, `MAX_TRAIN_STEPS = 2000`
- Full paper-style sweep: `RANKS = [1, 2, 4, 8, 16]`, `MAX_TRAIN_STEPS = None`

The paper's GPT-2 Medium/E2E sweep trained ranks for roughly 26k steps and found BLEU highest around `r=4`.

In [ ]:
RANKS = [1, 2, 4, 8, 16]
FIXED_ALPHA = 32
MAX_TRAIN_STEPS = None  # Set to 2000 for a quick pilot; None means full 5 epochs.
GENERATION_BATCH_SIZE = 16
RUN_TRAINING = True
RUN_GENERATION_AND_EVAL = True
RUN_OFFICIAL_E2E = True  # Optional; requires cloning external/e2e-metrics.

print('Ranks:', RANKS)
print('MAX_TRAIN_STEPS:', MAX_TRAIN_STEPS)
print('Generation batch size:', GENERATION_BATCH_SIZE)

## 7. Create Rank-Specific Configs

In [ ]:
import copy
import yaml
from pathlib import Path

base_config_path = Path('configs/e2e_gpt2_medium_lora.yaml')
base_config = yaml.safe_load(base_config_path.read_text())
rank_config_paths = []

for rank in RANKS:
    cfg = copy.deepcopy(base_config)
    run_name = f'e2e_lora_r{rank}_alpha{FIXED_ALPHA}'
    run_dir = f'outputs/runs/rank_sweep/{run_name}'

    cfg['project']['output_dir'] = run_dir
    cfg['lora']['rank'] = rank
    cfg['lora']['alpha'] = FIXED_ALPHA
    cfg['generation']['decoder'] = 'official_beam'
    cfg['generation']['length_penalty'] = 0.9
    cfg['generation']['batch_size'] = GENERATION_BATCH_SIZE
    cfg['evaluation']['predictions_file'] = f'{run_dir}/generations_test.txt'
    cfg['evaluation']['references_file'] = '../data/processed/e2e_gpt2/references_test.txt'

    out_path = Path(f'configs/rank_sweep/e2e_gpt2_medium_lora_r{rank}.yaml')
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')
    rank_config_paths.append((rank, out_path, Path(run_dir)))

for rank, config_path, run_dir in rank_config_paths:
    print(rank, config_path, '->', run_dir)

## 8. Sanity Checks

In [ ]:
import subprocess

subprocess.run(['python', '-m', 'pytest'], check=True)
for rank, config_path, run_dir in rank_config_paths:
    print('\n=== Rank', rank, 'parameter count ===')
    subprocess.run(
        ['python', 'scripts/count_params.py', '--config', str(config_path)],
        check=True,
    )

## 9. Train One Adapter Per Rank

This is the expensive cell. Each full rank run is a separate GPT-2 Medium LoRA training job.

In [ ]:
from collections import deque
import shlex
import subprocess


def run_checked(cmd):
    print('+', shlex.join(cmd), flush=True)
    tail = deque(maxlen=80)
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
        tail.append(line)
    returncode = process.wait()
    if returncode != 0:
        print(f'\nCommand failed with exit code {returncode}: {shlex.join(cmd)}')
        print('Last output lines:')
        print(''.join(tail))
        raise subprocess.CalledProcessError(returncode, cmd)


if RUN_TRAINING:
    for rank, config_path, run_dir in rank_config_paths:
        print(f'=== Training rank {rank} ===')
        cmd = [
            'python', 'scripts/train.py',
            '--config', str(config_path),
            '--train',
            '--device', 'cuda',
        ]
        if MAX_TRAIN_STEPS is not None:
            cmd += ['--max-train-steps', str(MAX_TRAIN_STEPS)]
        run_checked(cmd)
else:
    print('RUN_TRAINING=False; skipping training.')

## 10. Inspect Validation Records

In [ ]:
import json

for rank, config_path, run_dir in rank_config_paths:
    metrics_path = run_dir / 'metrics.jsonl'
    print(f'\n=== Rank {rank} validation ===')
    if not metrics_path.exists():
        print('missing:', metrics_path)
        continue
    records = [json.loads(line) for line in metrics_path.read_text().splitlines() if line.strip()]
    validation = [record for record in records if 'valid_loss' in record]
    if not validation:
        print('no validation records; did training stop before a full epoch?')
    for record in validation:
        print(record)

## 11. Generate And Evaluate Each Rank

In [ ]:
import os
import subprocess

if RUN_GENERATION_AND_EVAL:
    env = {**os.environ, 'TOKENIZERS_PARALLELISM': 'false', 'TRANSFORMERS_VERBOSITY': 'error'}
    for rank, config_path, run_dir in rank_config_paths:
        print(f'\n=== Generating/evaluating rank {rank} ===')
        adapter = run_dir / 'checkpoints' / 'adapter_final.pt'
        if not adapter.exists():
            raise FileNotFoundError(f'Missing adapter: {adapter}')

        subprocess.run([
            'python', 'scripts/generate.py',
            '--config', str(config_path),
            '--split', 'test',
            '--adapter', str(adapter),
            '--batch-size', str(GENERATION_BATCH_SIZE),
        ], check=True, env=env)

        subprocess.run(['python', 'scripts/evaluate.py', '--config', str(config_path)], check=True)
        subprocess.run([
            'python', 'scripts/make_figures.py',
            '--config', str(config_path),
            '--run-dir', str(run_dir),
            '--figures-dir', str(run_dir / 'figures'),
        ], check=True)
else:
    print('RUN_GENERATION_AND_EVAL=False; skipping generation/evaluation.')

## 12. Optional Official E2E Scorer

This runs the external E2E metric script on each rank's grouped `e2e_refs` / `e2e_preds` files and saves stdout/stderr into each run folder.

In [ ]:
import subprocess
from pathlib import Path

if RUN_OFFICIAL_E2E:
    Path('external').mkdir(exist_ok=True)
    if not Path('external/e2e-metrics/.git').exists():
        subprocess.run(['git', 'clone', 'https://github.com/tuetschek/e2e-metrics.git', 'external/e2e-metrics'], check=True)

    for rank, config_path, run_dir in rank_config_paths:
        ref_file = run_dir / 'generations_test.e2e_refs.txt'
        pred_file = run_dir / 'generations_test.e2e_preds.txt'
        out_file = run_dir / 'generations_test.official_e2e_metrics.txt'
        print(f'\n=== Official E2E metrics rank {rank} ===')
        result = subprocess.run(
            ['python', 'external/e2e-metrics/measure_scores.py', str(ref_file), str(pred_file), '-p'],
            text=True,
            capture_output=True,
        )
        out_file.write_text(
            'STDOUT:\n' + result.stdout + '\n\nSTDERR:\n' + result.stderr,
            encoding='utf-8',
        )
        print(result.stdout)
        if result.returncode != 0:
            print(result.stderr)
else:
    print('RUN_OFFICIAL_E2E=False; skipping official scorer.')

## 13. Build Rank Sweep Summary

In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

rows = []

for rank, config_path, run_dir in rank_config_paths:
    metrics_path = run_dir / 'generations_test.metrics.json'
    train_metrics_path = run_dir / 'metrics.jsonl'
    params_path = run_dir / 'parameter_report.json'

    metrics = json.loads(metrics_path.read_text()) if metrics_path.exists() else {}
    params = json.loads(params_path.read_text()) if params_path.exists() else {}

    validation_records = []
    if train_metrics_path.exists():
        for line in train_metrics_path.read_text().splitlines():
            if line.strip():
                record = json.loads(line)
                if 'valid_loss' in record:
                    validation_records.append(record)

    best_valid = min(validation_records, key=lambda r: r['valid_loss']) if validation_records else {}
    final_valid = validation_records[-1] if validation_records else {}

    rows.append({
        'rank': rank,
        'alpha': FIXED_ALPHA,
        'trainable_params': params.get('trainable'),
        'best_valid_loss': best_valid.get('valid_loss'),
        'best_valid_ppl': best_valid.get('valid_ppl'),
        'best_valid_nll_loss': best_valid.get('valid_nll_loss'),
        'best_valid_nll_ppl': best_valid.get('valid_nll_ppl'),
        'best_valid_epoch': best_valid.get('epoch'),
        'final_valid_loss': final_valid.get('valid_loss'),
        'final_valid_nll_loss': final_valid.get('valid_nll_loss'),
        'bleu': metrics.get('bleu'),
        'rouge_l': metrics.get('rouge_l'),
        'line_bleu': metrics.get('line_bleu'),
        'line_rouge_l': metrics.get('line_rouge_l'),
        'run_dir': str(run_dir),
    })

df = pd.DataFrame(rows).sort_values('rank')
summary_path = Path('outputs/runs/rank_sweep_summary.csv')
summary_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(summary_path, index=False)
display(df)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

if 'best_valid_loss' in df and df['best_valid_loss'].notna().any():
    axes[0].plot(df['rank'], df['best_valid_loss'], marker='o')
axes[0].set_xscale('log', base=2)
axes[0].set_xticks(df['rank'])
axes[0].set_xticklabels([str(r) for r in df['rank']])
axes[0].set_xlabel('LoRA rank r')
axes[0].set_ylabel('Best validation loss')
axes[0].set_title('Validation Loss vs Rank')
axes[0].grid(alpha=0.25)

if 'bleu' in df and df['bleu'].notna().any():
    axes[1].plot(df['rank'], df['bleu'], marker='o', label='BLEU')
if 'rouge_l' in df and df['rouge_l'].notna().any():
    axes[1].plot(df['rank'], df['rouge_l'], marker='o', label='ROUGE-L')
axes[1].set_xscale('log', base=2)
axes[1].set_xticks(df['rank'])
axes[1].set_xticklabels([str(r) for r in df['rank']])
axes[1].set_xlabel('LoRA rank r')
axes[1].set_ylabel('Score')
axes[1].set_title('Test Metrics vs Rank')
axes[1].legend()
axes[1].grid(alpha=0.25)

fig.tight_layout()
plot_path = Path('outputs/runs/rank_sweep_summary.png')
fig.savefig(plot_path, dpi=180)

print('Saved:', summary_path)
print('Saved:', plot_path)

## 14. Back Up Rank Sweep To Drive

In [ ]:
from pathlib import Path
import shutil

DRIVE_SWEEP_DIR.mkdir(parents=True, exist_ok=True)

for rank, config_path, run_dir in rank_config_paths:
    if run_dir.exists():
        destination = DRIVE_SWEEP_DIR / run_dir.name
        shutil.copytree(run_dir, destination, dirs_exist_ok=True)
        print('Backed up:', run_dir, '->', destination)
    if config_path.exists():
        config_dest = DRIVE_SWEEP_DIR / 'configs' / config_path.name
        config_dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(config_path, config_dest)

for path in [
    Path('outputs/runs/rank_sweep_summary.csv'),
    Path('outputs/runs/rank_sweep_summary.png'),
    Path('colab_rank_sweep_lora.ipynb'),
]:
    if path.exists():
        shutil.copy2(path, DRIVE_SWEEP_DIR / path.name)
        print('Backed up:', path)

!find /content/drive/MyDrive/lora_rank_sweep -maxdepth 3 -type f | sort | tail -80